# Faruq-v3 — Scale-Identifiability Audit

Validation-only diagnostic. **No training, no inference, no test.** This audit measures whether frozen hard pairs whose labels differ by `kecil/sedang/besar` are actually separable by apparent bbox scale in the image. It reuses the existing CPE0/CIR0 object-events only to mark pair errors.

Important: apparent image scale is not physical millimetre size. Overlap can demonstrate limited identifiability under the acquisition protocol, but it does not by itself prove a wrong label or grading rule.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-circle-cpe-hard-confusion-reduction-v1/events/CPE0_seed42_events.json',
    'experiments/faruq-v3-circle-cpe-hard-confusion-reduction-v1/events/CIR0_seed42_events.json',
    'experiments/faruq-v3-cross-model-hard-confusion-consensus-seed42-v1/cross_model_hard_confusion_consensus.json',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
CPE0_EVENT = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
CIR0_EVENT = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
CONSENSUS = require_project_artifact(PROJECT_ROOT, REQUIRED[3])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT/'faruq_grouped_summary.json').is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT/'experiments/faruq-v3-scale-identifiability-audit-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY = OUTPUT_ROOT/'scale_identifiability_audit.json'
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_scale_identifiability_audit.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
command = [
    sys.executable, '-m', 'coffee_detector.analysis.scale_identifiability_audit',
    '--cpe0-event', str(CPE0_EVENT), '--cir0-event', str(CIR0_EVENT),
    '--consensus-json', str(CONSENSUS), '--data-root', str(DATA_ROOT),
    '--output-root', str(OUTPUT_ROOT),
]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
assert result['screening_decision_remains'] == 'STOP_CIRCLE_CPE'


In [ ]:
import pandas as pd
from IPython.display import Image, display

rows = []
for pair in result['pairs']:
    L = pair['features']['long_side_norm']
    A = pair['features']['area_norm']
    rows.append({
        'family': pair['family'], 'n': pair['gt_instances_in_pair'],
        'low': pair['low_class'], 'high': pair['high_class'],
        'long_med_low': L['low_class']['median'], 'long_med_high': L['high_class']['median'],
        'long_AUC_expected': L['expected_order_auc'],
        'long_best_BA': L['best_single_threshold_balanced_accuracy_expected_direction'],
        'long_IQR_overlap': L['iqr_overlap']['fraction_of_iqr_union'],
        'area_med_low': A['low_class']['median'], 'area_med_high': A['high_class']['median'],
        'area_AUC_expected': A['expected_order_auc'],
        'area_best_BA': A['best_single_threshold_balanced_accuracy_expected_direction'],
        'area_IQR_overlap': A['iqr_overlap']['fraction_of_iqr_union'],
        'CPE0_pair_errors': pair['cpe0_pair_errors'], 'CIR0_pair_errors': pair['cir0_pair_errors'],
        'CPE0_err_in_long_overlap': L['error_overlap'].get('cpe0_pair_error_overlap_rate', 0.0),
        'CIR0_err_in_long_overlap': L['error_overlap'].get('cir0_pair_error_overlap_rate', 0.0),
    })
display(pd.DataFrame(rows))
print('FROZEN SIZE HARD PAIRS:', result['n_frozen_size_hard_pairs'])
print('SCREENING DECISION REMAINS:', result['screening_decision_remains'])
print('SUMMARY:', SUMMARY)
for pair in result['pairs']:
    print('\n###', pair['family'])
    for feature in ('long_side_norm','area_norm'):
        path = pair['plots'][feature]
        display(Image(filename=path, width=700))
